# label_integration.ipynb
## Integración de etiquetas clínicas en el dataset acústico
### TFG — Análisis de biomarcadores acústicos en ELA

**Autor:** Jakub Wysocki  

---

Integra la base de datos clínica (`clinical_data.xlsx`) con el
dataset acústico fusionado (`dataset_raw_fusionado.csv`).

**Lógica:**
- El Excel ya contiene exactamente los 63 sujetos del estudio (45 ELA + 18 controles).
- El CSV acústico tiene 68 filas (MATLAB procesó todas las carpetas disponibles).
- Se construye un `label_map` desde el Excel y se aplica al CSV.
- Las filas del CSV sin entrada en el `label_map` quedan con `NaN` y se descartan con `dropna`.
- **El Excel no se modifica en ningún momento.**

**Resultado:** `dataset_final.csv` — 63 filas, etiquetas completas, listo para EDA y ML.

## 0. Importaciones y rutas

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd

PROJECT_ROOT  = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.label_loader import run_clinical_pipeline, SUBJECT_ORDER
from src.preprocessing import assign_labels, save_dataset, validate_dataset

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_FUSED_CSV = PROCESSED_DIR / 'dataset_raw_fusionado.csv'
CLINICAL_XLSX = Path(r'C:\TFG\data\clinical_data.xlsx')

print(f'Excel clínico    : {CLINICAL_XLSX}  → existe: {CLINICAL_XLSX.exists()}')
print(f'CSV acústico     : {RAW_FUSED_CSV}  → existe: {RAW_FUSED_CSV.exists()}')
print(f'SUBJECT_ORDER    : {len(SUBJECT_ORDER)} entradas  [{SUBJECT_ORDER[0]} ... {SUBJECT_ORDER[-1]}]')

## 1. Pipeline clínico → label_map

In [ ]:
label_map = run_clinical_pipeline(CLINICAL_XLSX)

print(f'label_map: {len(label_map)} entradas  (esperado: 63)')

dist_clin = Counter(v['label_clinico'] for v in label_map.values())
dist_maq  = Counter(v['label_maquina']  for v in label_map.values())
print('\nlabel_clinico:', dict(sorted(dist_clin.items())))
print('label_maquina :', dict(sorted(dist_maq.items())))

## 2. Integración con el dataset acústico

In [ ]:
df_acoustic = pd.read_csv(RAW_FUSED_CSV)
print(f'CSV acústico: {df_acoustic.shape}  (esperado: (68, 204))')

In [ ]:
# Asignar IDs MATLAB reales — SUBJECT_ORDER define el orden fila a fila
assert len(SUBJECT_ORDER) == len(df_acoustic), (
    f'SUBJECT_ORDER ({len(SUBJECT_ORDER)}) ≠ filas CSV ({len(df_acoustic)})'
)
df_acoustic['subject_id'] = SUBJECT_ORDER
print(f'Primeros 5 IDs: {df_acoustic["subject_id"].head().tolist()}')
print(f'Últimos  5 IDs: {df_acoustic["subject_id"].tail().tolist()}')

In [ ]:
# Aplicar label_map — filas sin entrada quedan con NaN
df_labeled = assign_labels(df_acoustic, label_map)

nan_count = df_labeled['label_clinico'].isna().sum()
print(f'Filas con NaN en etiquetas: {nan_count}')
if nan_count > 0:
    print('IDs sin etiqueta (no presentes en el Excel clínico):')
    print(df_labeled.loc[df_labeled['label_clinico'].isna(), 'subject_id'].tolist())

In [ ]:
# Conservar solo las filas con etiqueta válida (join natural con el Excel)
df_final = df_labeled.dropna(
    subset=['label_clinico', 'label_maquina']
).reset_index(drop=True)

print(f'Dataset final: {df_final.shape}  (esperado: (63, 204))')

## 3. Verificación

In [ ]:
print('=== METADATOS ===')
print(df_final[['subject_id', 'genero', 'label_clinico', 'label_maquina']].head(15).to_string())

print('\n=== label_clinico ===')
print(df_final['label_clinico'].value_counts().to_string())
print('\n=== label_maquina ===')
print(df_final['label_maquina'].value_counts().to_string())

diff = df_final['label_clinico'] != df_final['label_maquina']
print(f'\nSujetos reetiquetados (clínico ≠ máquina): {diff.sum()}')
if diff.any():
    print(df_final.loc[diff, ['subject_id', 'label_clinico', 'label_maquina']].to_string(index=False))

In [ ]:
validate_dataset(df_final, expected_rows=63)
all_ok = df_final[['label_clinico', 'label_maquina']].isna().sum().sum() == 0
print('\n✓ Dataset final correcto. Listo para EDA y ML.' if all_ok else '\n⚠ Revisar etiquetas.')

## 4. Guardado

In [ ]:
save_dataset(df_final, PROCESSED_DIR, 'dataset_final.csv')
print(f'✓ dataset_final.csv  {df_final.shape}')
print('→ Siguiente: notebooks/eda.ipynb')